In [0]:
spark.sql("USE globalretail_silver")
spark.sql("""
          CREATE TABLE IF NOT EXISTS silver_products (
              product_id STRING,
              name STRING,
              category STRING,
              brand STRING,
              price DOUBLE,
              stock_quantity INT,
              rating DOUBLE,
              is_active BOOLEAN,
              price_category STRING,
              stock_status STRING,
              last_updated TIMESTAMP
          )
          USING DELTA
          """)

In [0]:
from datetime import datetime
last_processed_df = spark.sql("select max(last_updated) as last_updated from silver_products")
last_processed_timestamp = last_processed_df.collect()[0]['last_updated']

if last_processed_timestamp is None:
  last_processed_timestamp = datetime(1900, 1, 1)

In [0]:
#Create a temporary view of incremental bronze data
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW bronze_incremental_product AS
SELECT * 
FROM globalretail_bronze.bronze_products where ingestion_timestamp > '{last_processed_timestamp}'
""")

In [0]:
display(spark.sql("SELECT * FROM bronze_incremental_product"))

Data Transformation:
Price normalization (setting negative prices to 0)
Stock quantity normalization (setting negative stock to 0)
Rating normalization (clamping between 0 to 5)
Price categorization (Premium>1000, Standard>100, Budget)
Stock status calculation (Out of Stock, Low Stock>0, Moderate Stock>10, Sufficient Stock>50)

In [0]:
#Create a temporary view of incremental data from bronze temporary view
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW silver_incremental_product AS
SELECT
    product_id,
    name,
    category,
    brand,
    CASE
        WHEN price < 0 THEN 0
        ELSE price
    END AS price,
    CASE
        WHEN stock_quantity < 0 THEN 0
        ELSE stock_quantity
    END AS stock_quantity,
    CASE
        WHEN rating < 0 THEN 0
        WHEN rating > 5 THEN 5
        ELSE rating
    END AS rating,
    is_active,
    CASE
        WHEN price < 1000 THEN 'Premium'
        WHEN price < 100 THEN 'Standard'
        ELSE 'Budget'
    END AS price_category, 
    CASE
        WHEN stock_quantity = 0 THEN 'Out of Stock'
        WHEN stock_quantity <= 10 THEN 'Low Stock'
        WHEN Stock_quantity <= 50 THEN 'Medium Stock'
        ELSE 'High Stock'
    END AS stock_status,
    CURRENT_TIMESTAMP() last_updated
FROM bronze_incremental_product
WHERE name IS NOT NULL AND category IS NOT NULL
    """)

In [0]:
spark.sql("""
MERGE INTO silver_products t
USING silver_incremental_product s
ON t.product_id = s.product_id
WHEN MATCHED THEN
  UPDATE SET *
WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
display(spark.sql("SELECT * FROM silver_products"))